# Import Libraries

In [1]:
# Import necessary libraries for data manipulation, date and time operations,
# working with Excel files, handling zip files, and moving files.
import pandas as pd
import numpy as np
from datetime import datetime
import pytz
import openpyxl
from zipfile import ZipFile

# Import modules for interacting with the operating system and moving files.
import os
import shutil  # Import shutil for moving files

# Define Local Timezone

In [2]:
# Define the local timezone as 'Asia/Jakarta'. Replace with your desired time zone.
local_tz = pytz.timezone('Asia/Jakarta')
# Get the current date and time in the specified local timezone.
now_local = datetime.now(local_tz)
# Define the desired date format (Day-Month-Year).
date_format = "%d-%m-%Y" # Choose your desired format
# Format the current local date as a string using the defined format.
date_str = now_local.strftime(date_format)

# Import Data

In [3]:
# Import the 'files' module from google.colab to enable file uploads.
from google.colab import files
# Prompt the user to upload a file and store the uploaded file information.
file = files.upload()

Saving 1 April - MASTER SHEET.csv to 1 April - MASTER SHEET.csv


In [4]:
# Define the directory where uploaded files are stored in Colab.
upload_dir = '/content/'

try:
    # Find all files in the upload directory that end with '.csv'.
    csv_files = [file for file in os.listdir(upload_dir) if file.endswith('.csv')]

    # Check the number of CSV files found.
    if len(csv_files) == 1:
        # If exactly one CSV file is found, get its filename and full path.
        csv_filename = csv_files[0]
        file_path = os.path.join(upload_dir, csv_filename)

        # Print a message indicating the found CSV file.
        print(f"Found CSV file: {csv_filename}")

        # Read the CSV file into a pandas DataFrame.
        df = pd.read_csv(file_path)
        # Print a success message after reading the file.
        print("Successfully read the CSV file.")


    elif len(csv_files) == 0:
        # If no CSV files are found, print an error message.
        print("Error: No CSV files were found in the directory.")

    else:
        # If multiple CSV files are found, print an error message listing the files.
        print(f"Error: Multiple CSV files found. Please specify which one to use: {csv_files}")
        # You could add logic here to pick the newest file, or ask the user to choose.

# Handle the case where the specified directory does not exist.
except FileNotFoundError:
    print(f"Error: The directory '{upload_dir}' does not exist.")
# Handle any other unexpected errors that might occur.
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Found CSV file: 1 April - MASTER SHEET.csv
Successfully read the CSV file.


# Data Processing

In [5]:
# Create a deep copy of the original DataFrame for backup purposes.
df_backup = df.copy(deep=True)

# Convert total_price data type as integer
df['total_price'] = df['total_price'].fillna(0).astype(str).str.replace(',', '', regex=False).astype(float).astype(int)

# Define a list of desired column names for the DataFrame.
df_columns = [
    'driver_name', 'district', 'customer_name', 'delivery_date',
    'address', 'address_note', 'order_no', 'packaging_option',
    'distance_in_km', 'hubs', 'total_price', 'external_note',
    'internal_note', 'customer_note', 'time_slot', 'no_plastic',
    'payment_method', 'latitude', 'longitude', 'shipping_number (Box #)'
]

# Filtered dataframe with selected columns
df = df[df_columns]

# Sort the DataFrame by 'hubs', 'driver_name', and 'customer_name' in ascending order.
df = df.sort_values(by=['hubs', 'driver_name', 'customer_name'], ascending=[True, True, True])

# Display the first few rows of the sorted DataFrame.
display(df.head())

,driver_name,district,customer_name,delivery_date,address,address_note,order_no,packaging_option,distance_in_km,hubs,total_price,external_note,internal_note,customer_note,time_slot,no_plastic,payment_method,latitude,longitude,shipping_number (Box #)
0,KOP-DPK-ADEAPRIYANTI,kecamatan cinere,Afra Hollia,2026-04-01,"Jl. Puncak Pesanggrahan II No.19, Cinere, Kec....",Nomor 19,DH-32YE3IJDULXZ-NR,NaN,4.36,Beji - Depok,262100,NaN,NaN,NaN,slot-0,0,VA BCA,-6.34,106.78,8
1,KOP-DPK-ADEAPRIYANTI,kecamatan jagakarsa,Annisa,2026-04-01,Warung Mba Ika no. 46,NaN,PG-0BCJXI0Y0HZV-NR,HALF MEDIUM BOX,1.94,Beji - Depok,185709,NaN,NaN,NaN,slot-1,0,VA BCA,-6.35,106.80,103
2,KOP-DPK-ADEAPRIYANTI,kecamatan jagakarsa,Berlianna Indah Permata,2026-04-01,Masuk dari tanah kosong belakang warkop berkah...,NaN,PG-YKXVYIXAMJXB-NR,HALF MEDIUM BOX,3.65,Beji - Depok,78200,NaN,NaN,NaN,slot-1,0,GO-PAY Wallet,-6.34,106.80,111
3,KOP-DPK-ADEAPRIYANTI,kecamatan cinere,Dahlia,2026-04-01,No.130,WA saat sampai,PG-G78BIIKDOD0D-NR,HALF MEDIUM BOX,4.80,Beji - Depok,91200,NaN,NaN,NaN,slot-1,0,VA BCA,-6.33,106.78,33
4,KOP-DPK-ADEAPRIYANTI,kecamatan cinere,Diana Putri,2026-04-01,"Jalan Bukit Cinere Raya No.Kav. 171D, Gandul, ...","Lavanya Hills Residence,Cluster Ayana2, NoA12,...",DH-6X3BJIZCWI0D-NR,PLASTIC,3.92,Beji - Depok,246700,NaN,NaN,NaN,slot-0,0,GO-PAY Wallet,-6.34,106.79,35


# B2B and COD Payment Formatting

In [6]:
# Define a function to apply conditional formatting to rows.
def highlight_b2b_and_payment(row):
  """
  Applies conditional formatting to a DataFrame row based on 'time_slot' and 'payment_method'.
  - Rows with specific 'time_slot' values ('slot-0bb', 'slot-1bb', 'slot-b2b-2') will have a light blue background.
  - Rows with 'Cash on Delivery' as 'payment_method' will have green text.
  - All cells will have a black border.
  """

  # Get the values from the 'time_slot' and 'payment_method' columns for the current row.
  time_slot_value = row.loc['time_slot']
  payment_method_value = row.loc['payment_method']

  # Define default background and text colors.
  background_color = '#FFFFFF'  # Default color
  text_color = '#000000'  # Default text color

  # Apply light blue background if the time slot is one of the specified 'b2b' slots.
  if time_slot_value in ('slot-0bb', 'slot-1bb', 'slot-b2b-2'):
    background_color = '#7393B3'  # Light blue for specific time slots
  # Apply green text color if the payment method is 'Cash on Delivery'.
  elif payment_method_value == 'Cash on Delivery':
    text_color = 'green'  # Yellow for Cash on Delivery

# Return a list of styles for each cell in the row, including background color, text color, and a black border.
  return [
      f'background-color: {background_color}; color: {text_color}; border: 0.5px solid black' for r in row
  ]

# Apply the conditional formatting function to the DataFrame and set table styles for borders.
formatted_df = df.style \
  .apply(highlight_b2b_and_payment, axis=1) \
  .set_table_styles([
      {'selector': '', 'props': [('border', '0.5px solid black')]}  # Add border to all cells
  ])

# Export the formatted DataFrame to an Excel file named "formated_df.xlsx", without the index.
formatted_df.to_excel("formated_df.xlsx", index=False)

# 3PL List Delivery by Time Slot

In [7]:
def template_3pl_df(vendor_name, driver_name, time_slot):
  """
  Filters the DataFrame based on vendor name, driver name, and time slot,
  formats the filtered data, exports it to an Excel file, and moves the file
  to a designated 'exported_files' folder.

  Args:
      vendor_name (str): The name of the vendor.
      driver_name (str): The driver name or a pattern to match driver names.
      time_slot (str): The time slot or a pattern to match time slots.
  """
  # Extract rows from the DataFrame where the 'driver_name' column contains 'BLITZ-', 'blitz-', '3W', or '3w' (case-insensitive).
  filtered_df = df[df['driver_name'].str.contains(driver_name, case=True) & df['time_slot'].str.contains(time_slot, case=True)]

  get_time_slot = time_slot.split('-')[1][0]

  # Create a filename for the output Excel file including the current date and SLOT-0.
  filename = f"{vendor_name} List Delivery {date_str} SLOT-{get_time_slot}.xlsx"

  # Apply conditional formatting and borders to the filtered DataFrame using the previously defined function.
  formatted_df = filtered_df.style \
    .apply(highlight_b2b_and_payment, axis=1) \
    .set_table_styles([
        {'selector': '', 'props': [('border', '0.5px solid black')]}  # Add border to all cells
    ])

  # Export the filtered DataFrame to an Excel file if it's not empty.
  if not filtered_df.empty:
    formatted_df.to_excel(filename, index=False)

    # Print a success message indicating the file export.
    print(f"{filename} is exported succesfully")

    # Define the destination folder for the exported file.
    destination_folder = "/content/exported_files"  # Replace with your desired folder path
    # Create the destination folder if it doesn't exist.
    os.makedirs(destination_folder, exist_ok=True)
    # Move the exported file to the destination folder.
    shutil.move(filename, destination_folder)

    # Print a message indicating where the file was moved.
    print(f"File moved to: {os.path.join(destination_folder, filename)}")
  else:
    # If the DataFrame is empty, print a message indicating that no data was found.
    print(f"No data for {filename}. Skipping export.")

## Blitz

In [8]:
template_3pl_df(vendor_name='Blitz', driver_name='BLITZ-|blitz-|3W|3w', time_slot='slot-0|slot-0bb')

Blitz List Delivery 01-04-2026 SLOT-0.xlsx is exported succesfully
File moved to: /content/exported_files/Blitz List Delivery 01-04-2026 SLOT-0.xlsx


In [9]:
template_3pl_df(vendor_name='Blitz', driver_name='BLITZ-|blitz-|3W|3w', time_slot='slot-1|slot-1bb')

Blitz List Delivery 01-04-2026 SLOT-1.xlsx is exported succesfully
File moved to: /content/exported_files/Blitz List Delivery 01-04-2026 SLOT-1.xlsx


## Koperasi

In [10]:
template_3pl_df(vendor_name='Koperasi', driver_name='KOP-|kop-', time_slot='slot-0|slot-0bb')

Koperasi List Delivery 01-04-2026 SLOT-0.xlsx is exported succesfully
File moved to: /content/exported_files/Koperasi List Delivery 01-04-2026 SLOT-0.xlsx


In [11]:
template_3pl_df(vendor_name='Koperasi', driver_name='KOP-|kop-', time_slot='slot-1|slot-1bb')

Koperasi List Delivery 01-04-2026 SLOT-1.xlsx is exported succesfully
File moved to: /content/exported_files/Koperasi List Delivery 01-04-2026 SLOT-1.xlsx


## Dash

In [12]:
template_3pl_df(vendor_name='Dash', driver_name='DASH|dash', time_slot='slot-0|slot-0bb')

Dash List Delivery 01-04-2026 SLOT-0.xlsx is exported succesfully
File moved to: /content/exported_files/Dash List Delivery 01-04-2026 SLOT-0.xlsx


In [13]:
template_3pl_df(vendor_name='Dash', driver_name='DASH|dash', time_slot='slot-1|slot-1bb')

Dash List Delivery 01-04-2026 SLOT-1.xlsx is exported succesfully
File moved to: /content/exported_files/Dash List Delivery 01-04-2026 SLOT-1.xlsx


# 3PL Support Function

In [14]:
def template_3pl_support_df(vendor_supported, vendor_name, driver_name, time_slot, hubs):
  """
  Filters the DataFrame for 3PL support based on vendor supported, vendor name, driver name,
  time slot, and hubs. Formats the filtered data, exports it to an Excel file,
  and moves the file to a designated 'exported_files' folder.

  Args:
      vendor_supported (str): The name of the vendor being supported.
      vendor_name (str): The name of the vendor.
      driver_name (str): The driver name or a pattern to match driver names.
      time_slot (str): The time slot or a pattern to match time slots.
      hubs (str): The hubs or a pattern to match hubs.
  """
  # Extract rows from the DataFrame where the 'driver_name' column contains 'BLITZ-', 'blitz-', '3W', or '3w' (case-insensitive).
  filtered_df = df[df['driver_name'].str.contains(driver_name, case=True) & df['time_slot'].str.contains(time_slot, case=True) & df['hubs'].str.contains(hubs, case=True)]

  get_time_slot = time_slot.split('-')[1][0]

  # Create a filename for the output Excel file including the current date and SLOT-0.
  filename = f"[{vendor_supported}] Support {vendor_name} List Delivery {date_str} SLOT-{get_time_slot}.xlsx"

  # Apply conditional formatting and borders to the filtered DataFrame using the previously defined function.
  formatted_df = filtered_df.style \
    .apply(highlight_b2b_and_payment, axis=1) \
    .set_table_styles([
        {'selector': '', 'props': [('border', '0.5px solid black')]}  # Add border to all cells
    ])

  # Export the filtered DataFrame to an Excel file if it's not empty.
  if not filtered_df.empty:
    formatted_df.to_excel(filename, index=False)

    # Print a success message indicating the file export.
    print(f"{filename} is exported succesfully")

    # Define the destination folder for the exported file.
    destination_folder = "/content/exported_files"  # Replace with your desired folder path
    # Create the destination folder if it doesn't exist.
    os.makedirs(destination_folder, exist_ok=True)
    # Move the exported file to the destination folder.
    shutil.move(filename, destination_folder)

    # Print a message indicating where the file was moved.
    print(f"File moved to: {os.path.join(destination_folder, filename)}")
  else:
    # If the DataFrame is empty, print a message indicating that no data was found.
    print(f"No data for {filename}. Skipping export.")

In [15]:
template_3pl_support_df(vendor_supported='Koperasi', vendor_name='DASH', driver_name='DASH|dash', time_slot='slot-0|0bb', hubs='Pondok Kopi - JAKTIM|Bekasi - CIBUBUR')

No data for [Koperasi] Support DASH List Delivery 01-04-2026 SLOT-0.xlsx. Skipping export.


# Zip 3PL List Delivery Files

In [16]:
# Define a function to zip all files within a specified folder.
def zip_exported_files(folder_path, zip_filename):
  """Zips all files within a folder.

  Args:
      folder_path (str): Path to the folder containing the files.
      zip_filename (str): Name of the output zip file.
  """
  # Create a zip file in write mode.
  with ZipFile(zip_filename, 'w') as zipf:
    # Walk through the directory tree starting from the folder_path.
    for root, _, files in os.walk(folder_path):
      # Iterate through each file in the current directory.
      for file in files:
        # Get the full path of the file.
        file_path = os.path.join(root, file)
        # Write the file to the zip archive, maintaining the relative path within the zip.
        zipf.write(file_path, os.path.relpath(file_path, folder_path))

# Define the folder containing the exported files.
destination_folder = "/content/exported_files"  # Replace with your folder path
# Define the name for the output zip file.
zip_filename = "exported_files.zip"  # Replace with your desired zip filename

# Call the function to zip the exported files.
zip_exported_files(destination_folder, zip_filename)
# Print a confirmation message.
print(f"Folder '{destination_folder}' zipped to '{zip_filename}'")

# Download the created zip file.
files.download("exported_files.zip")

Folder '/content/exported_files' zipped to 'exported_files.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Export Cash on Delivery Orders File

In [17]:
# Define a list of columns to include in the filtered DataFrame for Cash on Delivery orders.
df_columns = ['driver_name', 'customer_name', 'delivery_date', 'hubs', 'order_no', 'total_price', 'time_slot', 'payment_method', 'shipping_number (Box #)']

# First, filter the rows based on the conditions.
df_filtered_rows = df[
    (df['payment_method'] == 'Cash on Delivery') &
    (df['time_slot'].isin(['slot-0', 'slot-1']))
]

# Then, select the desired columns from the row-filtered DataFrame.
df_filtered_cod = df_filtered_rows[df_columns]

In [18]:
# Define a function to apply conditional formatting for Cash on Delivery orders.
def cash_on_delivery_orders(row):
  """
  Applies conditional formatting to a DataFrame row for Cash on Delivery orders.
  - Rows with 'Cash on Delivery' as 'payment_method' and 'slot-0' or 'slot-1' as 'time_slot' will have green text.
  - All cells will have a black border.
  """
  # Get the values for 'time_slot' and 'payment_method' for the current row.
  time_slot_value = row.loc['time_slot']
  payment_method_value = row.loc['payment_method']

  # Define the default text color.
  text_color = '#000000'  # Default text color

  # If the time slot is 'slot-0' or 'slot-1' and the payment method is 'Cash on Delivery', set text color to green.
  if time_slot_value in ('slot-0', 'slot-1') and payment_method_value == 'Cash on Delivery':
    text_color = 'green'  # Green for Cash on Delivery

  # Return a list of styles for each cell in the row, applying the text color and adding a black border.
  return [
      f'color: {text_color}; border: 0.5px solid black' for r in row
  ]
# Apply the conditional formatting function and set table styles for borders to the filtered DataFrame.
formatted_df = df_filtered_cod.style \
    .apply(cash_on_delivery_orders, axis=1) \
    .set_table_styles([
        {'selector': '', 'props': [('border', '0.5px solid black')]}  # Add border to all cells
    ])

# Export the formatted DataFrame to an Excel file named "Cash on Delivery Orders.xlsx", without the index.
formatted_df.to_excel("Cash on Delivery Orders.xlsx", index=False)

In [19]:
# Load the Excel workbook named "Cash on Delivery Orders.xlsx".
# First, load it with pandas to check the conditions.
try:
    temp_df_check = pd.read_excel("Cash on Delivery Orders.xlsx")
    time_slots_in_file = temp_df_check['time_slot'].unique()

    if 'slot-0' in time_slots_in_file and 'slot-1' in time_slots_in_file:
        print("Both 'slot-0' and 'slot-1' found. Proceeding with formatting and export.")
        workbook = openpyxl.load_workbook("Cash on Delivery Orders.xlsx")
        # Select the active worksheet in the workbook.
        sheet = workbook.active

        # Iterate through each row in the sheet.
        for row in sheet.iter_rows():
            # Iterate through each cell in the current row.
            for cell in row:
                # Set the horizontal and vertical alignment of the cell to center.
                cell.alignment = openpyxl.styles.Alignment(horizontal='center', vertical='center')

        # Auto-adjust the width of each column.
        for column in sheet.columns:
            # Initialize the maximum length found in the column to 0.
            max_length = 0
            # Get the column letter for the current column.
            column_letter = openpyxl.utils.get_column_letter(column[0].column)
            # Iterate through each cell in the column.
            for cell in column:
                try:
                    # Check if the length of the string representation of the cell's value is greater than the current max_length.
                    if len(str(cell.value)) > max_length:
                        # Update max_length if a longer value is found.
                        max_length = len(str(cell.value))
                except:
                    # Ignore any errors that occur while processing cell values.
                    pass
            # Calculate the adjusted width for the column based on the maximum length.
            adjusted_width = (max_length + 1.5)
            # Set the column dimension width in the sheet.
            sheet.column_dimensions[column_letter].width = adjusted_width

        # Save the modified workbook, overwriting the original file.
        workbook.save("Cash on Delivery Orders.xlsx")

        # Print a confirmation message.
        print("Text in 'Cash on Delivery Orders.xlsx' has been center-aligned")

        # Download the modified Excel file.
        files.download("Cash on Delivery Orders.xlsx")
    else:
        print("Condition not met: 'Cash on Delivery Orders.xlsx' does not contain both 'slot-0' and 'slot-1'. Waiving save and download.")

except FileNotFoundError:
    print("Error: 'Cash on Delivery Orders.xlsx' not found. It might not have been exported.")
except Exception as e:
    print(f"An error occurred: {e}")

Both 'slot-0' and 'slot-1' found. Proceeding with formatting and export.
Text in 'Cash on Delivery Orders.xlsx' has been center-aligned


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>